In [ ]:
# =====================================================================
# MODEL: Artificial Neural Network (ANN) - Multilayer Perceptron
# DATASET: data_without_weather.csv
# GOAL: Predict 'PROBABLE_CAUSE' (Classification)
# =====================================================================
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load Data
df = pd.read_csv('data_without_weather.csv')
df = df.dropna(subset=['PROBABLE_CAUSE'])

# Filter rare classes (< 5 samples) to ensure stable train/test splits
class_counts = df['PROBABLE_CAUSE'].value_counts()
valid_classes = class_counts[class_counts >= 5].index
df = df[df['PROBABLE_CAUSE'].isin(valid_classes)]

cols_to_drop = ['PROBABLE_CAUSE', 'DATEOFDTC', 'DTC_NO_JOB_NO', 'OBSERVATION_DURING_DTC', 'PLACE_OF_DAMAGED', 'FEEDER']
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
y_text = df['PROBABLE_CAUSE']

# 2. Target Encoding for Neural Network
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
num_classes = len(label_encoder.classes_)

# 3. Preprocessing Pipeline
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

# Note: sparse_output=False is critical for Keras to accept the array
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ("cat", Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# 5. Build ANN Model
ann_model = Sequential([
    Dense(256, activation="relu", input_shape=(X_train_prep.shape[1],)),
    BatchNormalization(), Dropout(0.3),
    Dense(128, activation="relu"),
    BatchNormalization(), Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

ann_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 6. Train Model
print("Training ANN on Data WITHOUT Weather...")
ann_model.fit(X_train_prep, y_train, epochs=30, batch_size=64, validation_split=0.2, callbacks=[early_stopping], verbose=1)

# 7. Evaluate
y_pred_probs = ann_model.predict(X_test_prep)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n--- ANN RESULTS (NO WEATHER) ---")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro Avg F1-Score:", f1_score(y_test, y_pred, average='macro'))
print("Weighted Avg F1-Score:", f1_score(y_test, y_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))